# LLM Support Ticket Classifier — Analysis

Full pipeline analysis: data overview, classification results, deflection analysis, and ROI modeling.

**Prerequisites:**
1. `python data/generate_tickets.py` — generates synthetic tickets
2. `python classifier.py` — runs LLM classification (requires `ANTHROPIC_API_KEY`)
3. Then run this notebook top to bottom

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

warnings.filterwarnings('ignore')

# Paths
BASE_DIR      = Path('.').resolve()
DATA_DIR      = BASE_DIR / 'data'
TICKETS_PATH  = DATA_DIR / 'tickets.csv'
CLASSIF_PATH  = DATA_DIR / 'tickets_classified.csv'
TAXONOMY_PATH = DATA_DIR / 'taxonomy.json'

# Style
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = sns.color_palette('Set2', 8)
sns.set_palette('Set2')

print('Notebook initialized. Loading data...')

In [ ]:
# Load data
df_raw = pd.read_csv(TICKETS_PATH)
df_raw['created_date'] = pd.to_datetime(df_raw['created_date'])

with open(TAXONOMY_PATH) as f:
    taxonomy = json.load(f)

CATEGORIES = taxonomy['categories']
PRIORITIES = taxonomy['priority_levels']
DEFLECTABLE_CATS = taxonomy['deflectable']

# Load classified data if available
classified_available = CLASSIF_PATH.exists()
if classified_available:
    df = pd.read_csv(CLASSIF_PATH)
    df['created_date'] = pd.to_datetime(df['created_date'])
    print(f'Loaded classified data: {len(df)} tickets')
else:
    df = df_raw.copy()
    print(f'NOTE: Classified data not found. Using raw tickets for section 1.')
    print(f'Run classifier.py to generate tickets_classified.csv')

print(f'Raw tickets: {len(df_raw)}')
print(f'Date range: {df_raw.created_date.min().date()} to {df_raw.created_date.max().date()}')

---
## 1. Data Overview

Distribution of the 300 synthetic support tickets across category, channel, priority, handle time, and CSAT.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Support Ticket Data Overview — 300 Synthetic Tickets', fontsize=14, fontweight='bold', y=1.01)

# 1a. Volume by Category
ax = axes[0, 0]
cat_counts = df_raw['category_actual'].value_counts()
colors = sns.color_palette('Set2', len(cat_counts))
bars = ax.barh(cat_counts.index, cat_counts.values, color=colors)
ax.set_title('Ticket Volume by Category', fontweight='bold')
ax.set_xlabel('Number of Tickets')
for bar, val in zip(bars, cat_counts.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val}', va='center', fontsize=9)
ax.set_xlim(0, cat_counts.max() * 1.15)

# 1b. Volume by Channel
ax = axes[0, 1]
ch_counts = df_raw['channel'].value_counts()
wedge_colors = sns.color_palette('Pastel1', len(ch_counts))
ax.pie(ch_counts.values, labels=ch_counts.index, autopct='%1.1f%%',
       colors=wedge_colors, startangle=90)
ax.set_title('Ticket Volume by Channel', fontweight='bold')

# 1c. Priority Distribution
ax = axes[1, 0]
pri_counts = df_raw['priority_actual'].value_counts().reindex(PRIORITIES, fill_value=0)
pri_colors = ['#2196F3', '#FF9800', '#F44336', '#9C27B0']
bars = ax.bar(pri_counts.index, pri_counts.values, color=pri_colors[:len(pri_counts)])
ax.set_title('Priority Distribution', fontweight='bold')
ax.set_ylabel('Number of Tickets')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', fontsize=10)

# 1d. Handle Time Distribution
ax = axes[1, 1]
for priority, color in zip(PRIORITIES, pri_colors):
    subset = df_raw[df_raw['priority_actual'] == priority]['handle_time_minutes']
    ax.hist(subset, bins=15, alpha=0.6, label=priority, color=color)
ax.set_title('Handle Time Distribution by Priority', fontweight='bold')
ax.set_xlabel('Handle Time (minutes)')
ax.set_ylabel('Count')
ax.legend()

plt.tight_layout()
plt.show()

print('\nSummary Statistics:')
print(df_raw[['handle_time_minutes', 'csat_score', 'resolved_first_contact']].describe().round(2))

In [ ]:
# CSAT Distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# CSAT overall
ax = axes[0]
csat_counts = df_raw['csat_score'].value_counts().sort_index()
csat_colors = ['#F44336', '#FF9800', '#FFC107', '#8BC34A', '#4CAF50']
bars = ax.bar(csat_counts.index, csat_counts.values, color=csat_colors)
ax.set_title('CSAT Score Distribution', fontweight='bold')
ax.set_xlabel('CSAT Score (1=Poor, 5=Excellent)')
ax.set_ylabel('Number of Tickets')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', fontsize=10)

# CSAT by category boxplot
ax = axes[1]
cat_order = df_raw.groupby('category_actual')['csat_score'].mean().sort_values(ascending=False).index
df_raw.boxplot(column='csat_score', by='category_actual',
               ax=ax, order=cat_order, grid=False,
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='red', linewidth=2))
ax.set_title('CSAT by Category', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('CSAT Score')
plt.sca(ax)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.suptitle('')

plt.tight_layout()
plt.show()

avg_csat = df_raw['csat_score'].mean()
fcr_rate = df_raw['resolved_first_contact'].mean()
print(f'Average CSAT: {avg_csat:.2f}/5.0')
print(f'First Contact Resolution: {fcr_rate:.1%}')

---
## 2. Classification Results

Accuracy metrics comparing Claude's predictions to actual labels.

In [ ]:
if not classified_available:
    print('[!]  Classification results not available.')
    print('Run: python classifier.py')
    print('\nThe notebook will use placeholder metrics for demonstration.')
    
    # Simulate plausible predictions for notebook demonstration
    import numpy as np
    rng = np.random.default_rng(42)
    
    # Simulate ~80% category accuracy, ~75% priority accuracy
    def simulate_prediction(actual, options, accuracy):
        if rng.random() < accuracy:
            return actual
        others = [o for o in options if o != actual]
        return rng.choice(others) if others else actual
    
    df['predicted_category'] = df['category_actual'].apply(
        lambda x: simulate_prediction(x, CATEGORIES, 0.80))
    df['predicted_priority'] = df['priority_actual'].apply(
        lambda x: simulate_prediction(x, PRIORITIES, 0.75))
    df['deflectable'] = df['predicted_category'].isin(DEFLECTABLE_CATS)
    df['confidence'] = rng.choice(['high', 'medium', 'low'], size=len(df), p=[0.60, 0.30, 0.10])
    df['reasoning'] = 'Simulated classification for demonstration.'
    print('\nUsing simulated predictions for visualization purposes.')
else:
    print(f'Using actual classification results from {CLASSIF_PATH.name}')

cat_acc = accuracy_score(df['category_actual'], df['predicted_category'])
pri_acc = accuracy_score(df['priority_actual'], df['predicted_priority'])
cat_f1  = f1_score(df['category_actual'], df['predicted_category'], average='weighted', zero_division=0)

print(f'\nCategory Accuracy:    {cat_acc:.1%}')
print(f'Category F1 (wtd):    {cat_f1:.1%}')
print(f'Priority Accuracy:    {pri_acc:.1%}')

print('\nDetailed Classification Report (Category):')
print(classification_report(df['category_actual'], df['predicted_category'],
                             labels=CATEGORIES, zero_division=0))

In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Category confusion matrix
ax = axes[0]
cm_cat = confusion_matrix(df['category_actual'], df['predicted_category'],
                           labels=CATEGORIES)
# Normalize
cm_cat_norm = cm_cat.astype(float) / cm_cat.sum(axis=1, keepdims=True)
short_labels = [c.replace(' & ', '\n& ').replace(' Request', '\nRequest') for c in CATEGORIES]
sns.heatmap(cm_cat_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=short_labels, yticklabels=short_labels,
            ax=ax, cbar_kws={'label': 'Proportion'})
ax.set_title('Category Confusion Matrix (Normalized)', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.tick_params(axis='y', rotation=0, labelsize=7)

# Priority confusion matrix
ax = axes[1]
cm_pri = confusion_matrix(df['priority_actual'], df['predicted_priority'],
                           labels=PRIORITIES)
cm_pri_norm = cm_pri.astype(float) / cm_pri.sum(axis=1, keepdims=True)
sns.heatmap(cm_pri_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=PRIORITIES, yticklabels=PRIORITIES,
            ax=ax, cbar_kws={'label': 'Proportion'})
ax.set_title('Priority Confusion Matrix (Normalized)', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')

plt.suptitle(f'Classification Accuracy — Category: {cat_acc:.1%} | Priority: {pri_acc:.1%}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Per-category F1 scores + confidence distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 by category
ax = axes[0]
report_dict = classification_report(
    df['category_actual'], df['predicted_category'],
    labels=CATEGORIES, output_dict=True, zero_division=0
)
f1_scores = {cat: report_dict.get(cat, {}).get('f1-score', 0) for cat in CATEGORIES}
cats_sorted = sorted(f1_scores, key=f1_scores.get, reverse=True)
f1_vals = [f1_scores[c] for c in cats_sorted]
colors = ['#4CAF50' if v >= 0.8 else '#FF9800' if v >= 0.6 else '#F44336' for v in f1_vals]
bars = ax.barh(cats_sorted, f1_vals, color=colors)
ax.axvline(0.8, color='green', linestyle='--', alpha=0.5, label='0.80 threshold')
ax.set_xlim(0, 1.05)
ax.set_title('F1 Score by Category', fontweight='bold')
ax.set_xlabel('F1 Score')
ax.legend(fontsize=9)
for bar, val in zip(bars, f1_vals):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9)

# Confidence distribution
ax = axes[1]
if 'confidence' in df.columns:
    conf_counts = df['confidence'].value_counts().reindex(['high', 'medium', 'low'], fill_value=0)
    conf_colors = ['#4CAF50', '#FF9800', '#F44336']
    bars = ax.bar(conf_counts.index, conf_counts.values, color=conf_colors)
    ax.set_title('Classifier Confidence Distribution', fontweight='bold')
    ax.set_ylabel('Number of Tickets')
    ax.set_xlabel('Confidence Level')
    for bar in bars:
        pct = bar.get_height() / len(df)
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{int(bar.get_height())}\n({pct:.0%})', ha='center', fontsize=10)
else:
    ax.text(0.5, 0.5, 'Confidence data\nnot available', ha='center', va='center',
            transform=ax.transAxes, fontsize=12)
    ax.set_title('Classifier Confidence Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 3. Deflection Analysis

Which tickets can be automatically deflected (resolved without human agent), and what does the distribution look like?

In [ ]:
# Overall deflection rate
total = len(df)
deflectable = int(df['deflectable'].sum())
non_deflectable = total - deflectable
deflection_rate = deflectable / total

print(f'Total tickets classified: {total}')
print(f'Deflectable:              {deflectable} ({deflection_rate:.1%})')
print(f'Human-required:           {non_deflectable} ({non_deflectable/total:.1%})')

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Pie chart
ax = axes[0]
ax.pie([deflectable, non_deflectable],
       labels=['Deflectable', 'Human Required'],
       colors=['#4CAF50', '#F44336'],
       autopct='%1.1f%%', startangle=90,
       explode=(0.05, 0),
       textprops={'fontsize': 11})
ax.set_title(f'Overall Deflection Rate\n{deflection_rate:.1%} deflectable', fontweight='bold')

# Deflection rate by category
ax = axes[1]
defl_by_cat = df.groupby('predicted_category')['deflectable'].agg(['sum', 'count'])
defl_by_cat['rate'] = defl_by_cat['sum'] / defl_by_cat['count']
defl_by_cat = defl_by_cat.sort_values('rate', ascending=True)
colors = ['#4CAF50' if r > 0.5 else '#FF9800' if r > 0.2 else '#F44336'
          for r in defl_by_cat['rate']]
bars = ax.barh(defl_by_cat.index, defl_by_cat['rate'], color=colors)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlim(0, 1.1)
ax.set_title('Deflection Rate by Category', fontweight='bold')
ax.set_xlabel('Deflection Rate')
for bar, val in zip(bars, defl_by_cat['rate']):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.0%}', va='center', fontsize=9)

# Deflection by priority
ax = axes[2]
defl_by_pri = df.groupby('predicted_priority')['deflectable'].agg(['sum', 'count'])
defl_by_pri['rate'] = defl_by_pri['sum'] / defl_by_pri['count']
defl_by_pri = defl_by_pri.reindex(PRIORITIES, fill_value=0)
bar_colors = ['#2196F3', '#FF9800', '#F44336', '#9C27B0']
bars = ax.bar(defl_by_pri.index, defl_by_pri['rate'] * 100, color=bar_colors)
ax.set_title('Deflection Rate by Priority', fontweight='bold')
ax.set_ylabel('Deflection Rate (%)')
ax.set_ylim(0, 110)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.0f}%', ha='center', fontsize=10)

plt.suptitle('Deflection Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Sample deflectable tickets
print('Sample Deflectable Tickets (top 8):')
print('=' * 80)
deflectable_df = df[df['deflectable'] == True][['ticket_id', 'predicted_category', 'predicted_priority', 'confidence', 'subject']]
if 'confidence' in deflectable_df.columns:
    sample = deflectable_df.sort_values('confidence', ascending=True).head(8)
else:
    sample = deflectable_df.head(8)

for _, row in sample.iterrows():
    print(f"  [{row['ticket_id']}] {row['predicted_category']} | {row['predicted_priority']} | {row.get('confidence', 'N/A')}")
    print(f"    Subject: {row['subject'][:70]}")
    print()

---
## 4. ROI Model

Financial impact of automated ticket deflection using the actual classification results.

In [ ]:
import sys
sys.path.insert(0, str(BASE_DIR))
from roi_model import calculate_roi, sensitivity_analysis, print_roi_report

# Calculate ROI using classified data
metrics = calculate_roi(
    df=df,
    cost_per_human_ticket=15.00,
    cost_per_automated_ticket=1.50,
    monthly_ticket_volume=5_000,
)

print_roi_report(metrics)

In [ ]:
# ROI Summary Table
summary_data = {
    'Metric': [
        'Monthly Ticket Volume',
        'Deflection Rate',
        'Tickets Deflected (monthly)',
        'Tickets Human-Handled (monthly)',
        'Monthly Cost (current)',
        'Monthly Cost (with automation)',
        'Monthly Savings',
        'Annual Savings',
        'Implementation Cost (est.)',
        'Payback Period',
        'Year-1 ROI',
    ],
    'Value': [
        f"{metrics['monthly_ticket_volume']:,}",
        f"{metrics['deflection_rate_applied']:.1%}",
        f"{metrics['monthly_deflected']:,}",
        f"{metrics['monthly_human_handled']:,}",
        f"${metrics['monthly_cost_current']:,.2f}",
        f"${metrics['monthly_cost_with_automation']:,.2f}",
        f"${metrics['monthly_savings']:,.2f}",
        f"${metrics['annual_savings']:,.2f}",
        f"${metrics['implementation_cost_assumed']:,.2f}",
        f"{metrics['payback_period_months']:.1f} months",
        f"{metrics['roi_year_1']:.0f}%",
    ]
}

summary_df = pd.DataFrame(summary_data)
summary_df.set_index('Metric', inplace=True)
print(summary_df.to_string())

In [ ]:
# Sensitivity Analysis
sa_df = sensitivity_analysis(
    df=df,
    cost_per_human_ticket=15.00,
    cost_per_automated_ticket=1.50,
    volume_min=1000,
    volume_max=20000,
    volume_step=1000,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Annual savings vs volume
ax = axes[0]
ax.fill_between(sa_df['monthly_volume'], sa_df['annual_savings'], alpha=0.3, color='steelblue')
ax.plot(sa_df['monthly_volume'], sa_df['annual_savings'], 'o-', color='steelblue', linewidth=2)
ax.axhline(10_000, color='red', linestyle='--', alpha=0.7, label='Implementation cost ($10K)')

# Mark 5K volume point
at_5k = sa_df[sa_df['monthly_volume'] == 5000]
if not at_5k.empty:
    ax.axvline(5000, color='orange', linestyle=':', alpha=0.7, label='Current volume (5K/mo)')
    ax.annotate(f"${at_5k['annual_savings'].values[0]:,.0f}/yr",
                xy=(5000, at_5k['annual_savings'].values[0]),
                xytext=(6000, at_5k['annual_savings'].values[0] * 0.85),
                arrowprops=dict(arrowstyle='->', color='black'),
                fontsize=10, fontweight='bold')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_title('Annual Savings vs Monthly Ticket Volume', fontweight='bold')
ax.set_xlabel('Monthly Ticket Volume')
ax.set_ylabel('Annual Savings ($)')
ax.legend()

# Cost comparison: current vs automated
ax = axes[1]
ax.fill_between(sa_df['monthly_volume'], sa_df['annual_cost_current'],
                sa_df['annual_cost_with_automation'],
                alpha=0.3, color='green', label='Savings zone')
ax.plot(sa_df['monthly_volume'], sa_df['annual_cost_current'], 'r-o',
        linewidth=2, markersize=4, label='Cost (current)')
ax.plot(sa_df['monthly_volume'], sa_df['annual_cost_with_automation'], 'g-o',
        linewidth=2, markersize=4, label='Cost (with automation)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_title('Annual Support Cost: Current vs Automated', fontweight='bold')
ax.set_xlabel('Monthly Ticket Volume')
ax.set_ylabel('Annual Cost ($K)')
ax.legend()

plt.suptitle('ROI Sensitivity Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nAt 5,000 tickets/month: ${metrics['annual_savings']:,.0f} annual savings")
print(f"Payback period: {metrics['payback_period_months']:.1f} months")

---
## 5. Sample Classifications

Ten example tickets showing the classifier's predictions, confidence, and reasoning.

In [ ]:
# Show 10 representative examples — mix of categories and confidence levels
sample_cols = ['ticket_id', 'subject', 'category_actual', 'predicted_category',
               'priority_actual', 'predicted_priority', 'deflectable', 'confidence']

# Get 1-2 from each confidence level if available
if 'confidence' in df.columns:
    samples = []
    for conf in ['high', 'medium', 'low']:
        subset = df[df['confidence'] == conf]
        if len(subset) > 0:
            n = min(4 if conf == 'high' else 3 if conf == 'medium' else 3, len(subset))
            samples.append(subset.sample(n, random_state=42))
    sample_df = pd.concat(samples).head(10)
else:
    sample_df = df.sample(10, random_state=42)

print('10 Sample Classifications')
print('=' * 100)
for i, (_, row) in enumerate(sample_df.iterrows(), 1):
    cat_match = '[OK]' if row.get('category_actual') == row.get('predicted_category') else '[X]'
    pri_match = '[OK]' if row.get('priority_actual') == row.get('predicted_priority') else '[X]'
    defl = 'DEFLECTABLE' if row.get('deflectable') else 'human required'
    conf = row.get('confidence', 'N/A')
    
    print(f"\n{i:2}. [{row.get('ticket_id', '?')}] {row.get('subject', '')[:65]}")
    print(f"    Category:  Actual={row.get('category_actual','?'):<25} "
          f"Predicted={row.get('predicted_category','?'):<25} {cat_match}")
    print(f"    Priority:  Actual={row.get('priority_actual','?'):<25} "
          f"Predicted={row.get('predicted_priority','?'):<25} {pri_match}")
    print(f"    Deflectable: {defl:<20} Confidence: {conf}")
    if 'reasoning' in row and isinstance(row['reasoning'], str) and len(row['reasoning']) > 5:
        print(f"    Reasoning: {row['reasoning'][:90]}")

In [ ]:
# Final summary
print('\n' + '=' * 60)
print('PIPELINE SUMMARY')
print('=' * 60)
print(f'  Tickets classified:      {total:,}')
print(f'  Category accuracy:       {cat_acc:.1%}')
print(f'  Priority accuracy:       {pri_acc:.1%}')
print(f'  Weighted F1 (category):  {cat_f1:.1%}')
print(f'  Deflection rate:         {deflection_rate:.1%}')
print(f'  Annual savings (5K/mo):  ${metrics["annual_savings"]:,.0f}')
print(f'  Payback period:          {metrics["payback_period_months"]:.1f} months')
print('=' * 60)